## Milestone 3: Data Cleaning

In [50]:
# importing packages
import pandas as pd

df = pd.read_csv('DOHMH_New_York_City_Restaurant_Inspection_Results_20260426.csv')
print("Dataset loaded successfully with " + str(len(df)) + " rows.")

C:\Users\flipp\AppData\Local\Temp\ipykernel_3648\1715357563.py:4: DtypeWarning: Columns (0: PHONE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('DOHMH_New_York_City_Restaurant_Inspection_Results_20260426.csv')


Dataset loaded successfully with 296352 rows.


### Renaming columns to lowercase with underscores for easier access

In [51]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Renamed columns:")
print(df.columns.tolist())

Renamed columns:
['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone', 'cuisine_description', 'inspection_date', 'action', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade', 'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location']


### Removing placeholder rows which are registered in the system but not yet inspected. 

In [52]:
# counting rows with the placeholder date
placeholder_count = (df['inspection_date'] == '01/01/1900').sum()
print("Rows with placeholder date 01/01/1900: " + str(placeholder_count))

# removing the non-placeholder dates
df = df[df['inspection_date'] != '01/01/1900']
print("Rows remaining after removal: " + str(len(df)))

Rows with placeholder date 01/01/1900: 3477
Rows remaining after removal: 292875


### Finding invalid coordinates in NYC

In [53]:
invalid_coords = ((df['latitude'] == 0) | (df['longitude'] == 0)).sum()
print("Rows with invalid coordinates: " + str(invalid_coords))

df.loc[df['latitude'] == 0, 'latitude'] = pd.NA
df.loc[df['longitude'] == 0, 'longitude'] = pd.NA

Rows with invalid coordinates: 3039


### Removing invalid coordinates

In [47]:
before = len(df)
df['zipcode'] = df['zipcode'].astype('Int64') 
invalid_zips = df[(df['zipcode'] < 10000) | (df['zipcode'] > 11700)]['zipcode']
print("Invalid ZIP codes found: " + str(len(invalid_zips)))
print("Examples of invalid ZIPs: " + str(invalid_zips.unique()[:10]))

df = df[(df['zipcode'] >= 10000) & (df['zipcode'] <= 11700) | df['zipcode'].isna()]
print("Rows removed: " + str(before - len(df)))

Invalid ZIP codes found: 92
Examples of invalid ZIPs: <IntegerArray>
[7307, 8550, 7071, 11950, 7304, 7030, 7002, 11762]
Length: 8, dtype: Int64
Rows removed: 92


### Dropping the redundant column of "location", as it is just a combination of latitude and longitude

In [55]:
df = df.drop(columns=['location'])
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 26


### Summary of cleaned data

In [56]:
print("Final dataset shape: " + str(df.shape))
print("\nData types:")
print(df.dtypes)
print("\nSample of cleaned data:")
df.head(10)

Final dataset shape: (292875, 26)

Data types:
camis                      int64
dba                          str
boro                         str
building                     str
street                       str
zipcode                  float64
phone                     object
cuisine_description          str
inspection_date              str
action                       str
violation_code               str
violation_description        str
critical_flag                str
score                    float64
grade                        str
grade_date                   str
record_date                  str
inspection_type              str
latitude                 float64
longitude                float64
community_board          float64
council_district         float64
census_tract             float64
bin                      float64
bbl                      float64
nta                          str
dtype: object

Sample of cleaned data:


,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,record_date,inspection_type,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta
5,50082345,COMPTON'S,Queens,30-02,14 STREET,11102.0,9174750573,Sandwiches,05/14/2024,Violations were cited in the following area(s).,...,04/24/2026,Cycle Inspection / Re-inspection,40.770703,-73.930040,401.0,22.0,7900.0,4005658.0,4.005150e+09,QN71
12,50044599,HOT DOG CONCESSION,Manhattan,4,PENN PLAZA,10121.0,2124656273,Hotdogs/Pretzels,04/29/2023,No violations were recorded at the time of thi...,...,04/24/2026,Cycle Inspection / Initial Inspection,40.750655,-73.991944,105.0,3.0,10100.0,1082908.0,1.007810e+09,MN17
17,50053320,COCO FRESH TEA & JUICE,Queens,37-01B,82 STREET,NaN,7185339496,Coffee/Tea,06/02/2023,Violations were cited in the following area(s).,...,04/24/2026,Cycle Inspection / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,4.000000e+00,NaN
18,41705732,BENTON SUSHI,Manhattan,156,EAST 45 STREET,10017.0,2129229788,Japanese,05/04/2025,No violations were recorded at the time of thi...,...,04/24/2026,Inter-Agency Task Force / Initial Inspection,40.753048,-73.973926,106.0,4.0,9200.0,1036181.0,1.012990e+09,MN19
20,50141498,DMM BAKERY,Brooklyn,6802,BAY PARKWAY,11204.0,7183314372,Chinese,06/16/2025,No violations were recorded at the time of thi...,...,04/24/2026,Cycle Inspection / Initial Inspection,40.612100,-73.983252,311.0,47.0,25800.0,3135132.0,3.055800e+09,BK28
36,50002426,SHISO,Queens,NaN,JFK INTERNATIONAL AIRPORT,11430.0,6464775291,Japanese,03/16/2017,Violations were cited in the following area(s).,...,04/24/2026,Cycle Inspection / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,50110806,THE CENTURION LOUNGE,Queens,LAGUARDIA,AP HEADHOUSE,11371.0,7185549605,American,03/23/2023,Violations were cited in the following area(s).,...,04/24/2026,Pre-permit (Operational) / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,41585425,LUCKY GARDEN,Bronx,305,WYCKOFF AVENUE,NaN,7184181002,Chinese,12/16/2025,Violations were cited in the following area(s).,...,04/24/2026,Cycle Inspection / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,NaN
48,50037907,BARRILES,Queens,NaN,37TH AVE,11372.0,3476491511,Latin American,09/18/2024,Violations were cited in the following area(s).,...,04/24/2026,Cycle Inspection / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,50098716,EATALY NY LLC (KIOSK),Manhattan,999,BROADWAY,NaN,3124685154,Pizza,06/14/2024,Establishment re-opened by DOHMH.,...,04/24/2026,Cycle Inspection / Reopening Inspection,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
